In [57]:
import chromadb
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
import ollama
import re

In [58]:
chroma_client = chromadb.PersistentClient(path="CHROMA")
MODEL = "mistral"
CONTEXT  =""
QUESTION = ""
PERSONA = f""" 
Answer the question using only the context below.
If the context does not contain the answer, say "I don't know based on the document."

Context:
{CONTEXT}

Question: {QUESTION}"""

In [59]:
loader = PyPDFLoader("data/1901.08579v1.pdf")
loader = PyPDFLoader('data/2001.08361v1.pdf')
loader = PyPDFLoader('data/2312.10997v5.pdf')
loader = PyPDFLoader('data/w35046.pdf')
loader = PyPDFLoader('data/2508.14111v2.pdf')
loader = PyPDFLoader('data\Qi_et_al_2026_Towards_trustworthy_agentic_AI_a_comprehensive_survey_of_AcadAI8260.pdf')
pages = loader.load()

<>:6: SyntaxWarning: invalid escape sequence '\Q'
<>:6: SyntaxWarning: invalid escape sequence '\Q'
C:\Users\HP\AppData\Local\Temp\ipykernel_28136\414004142.py:6: SyntaxWarning: invalid escape sequence '\Q'
  loader = PyPDFLoader('data\Qi_et_al_2026_Towards_trustworthy_agentic_AI_a_comprehensive_survey_of_AcadAI8260.pdf')


In [60]:
def cleanText(text):
    text = re.sub(r'-\n(\w)', r'\1', text)
    text = text.replace('•', '\n- ')
    text = re.sub(r'[\t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

In [61]:
for page in pages:
    page.page_content = cleanText(page.page_content)

In [62]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap=100 ) 
chunks = splitter.split_documents(pages)

In [63]:
def createEmbedding(text):
    response = ollama.embed(
        model = 'nomic-embed-text-v2-moe',
        input = text,
    )
    return response.embeddings[0]

In [64]:
doc_embeddings = []
for chunk in chunks:
    chunk_embedding = createEmbedding(chunk.page_content)
    doc_embeddings.append(chunk_embedding)

In [65]:
ids = [f"10201979:{i}" for i in range(len(chunks))]
documents = [chunk.page_content for chunk in chunks]
metadatas = [{"source": chunk.metadata.get("source", " "),  }for chunk in chunks]
embeddings = doc_embeddings

In [66]:
print(len(ids), len(documents), len(metadatas), len(embeddings))  # all four equal
print(len(embeddings))   

526 526 526 526
526


In [67]:
collection_research = chroma_client.get_or_create_collection(name='Chroma')

In [68]:
len(embeddings)

526

In [69]:
collection_research.add(
    ids = ids, 
    documents = documents, 
    metadatas = metadatas, 
    embeddings =  embeddings 
)

In [70]:
response = collection_research.query(
    query_texts=["What is a transformative AI?"],
    n_results=2,
    query_embeddings = [createEmbedding("What is a transformative AI?")],
)

In [71]:
response

{'ids': [['10201979:5', '10201979:6']],
 'embeddings': None,
 'documents': [['administered since 2006 (Grace 2015), four of which were ultimately reported in academic outlets \n(Baum, Goertzel et al. 2011, Müller and Bostrom 2016, Grace, Salvatier et al. 2018, Walsh 2018). \nThe primary concept of interest in this survey is transformative AI. For the purposes of this study \nwe define transformative AI to be AI that significantly transforms society by replacing humans for \na large portion  (i.e., 50% or greater) of economically useful work. As such, transformative AI \nwould include notions of artificial g eneral intelligence (AGI), high -level machine intelligence \n(HLMI), human- level machine intelligence, human -level artificial intelligence (HLAI ) and \nmachine superintelligence 1, as well as narrow AI technologies that also meet this definition.',
   'machine superintelligence 1, as well as narrow AI technologies that also meet this definition. \nFurthermore, the concept of tra

In [72]:
response['documents']

[['administered since 2006 (Grace 2015), four of which were ultimately reported in academic outlets \n(Baum, Goertzel et al. 2011, Müller and Bostrom 2016, Grace, Salvatier et al. 2018, Walsh 2018). \nThe primary concept of interest in this survey is transformative AI. For the purposes of this study \nwe define transformative AI to be AI that significantly transforms society by replacing humans for \na large portion  (i.e., 50% or greater) of economically useful work. As such, transformative AI \nwould include notions of artificial g eneral intelligence (AGI), high -level machine intelligence \n(HLMI), human- level machine intelligence, human -level artificial intelligence (HLAI ) and \nmachine superintelligence 1, as well as narrow AI technologies that also meet this definition.',
  'machine superintelligence 1, as well as narrow AI technologies that also meet this definition. \nFurthermore, the concept of transformative AI described here is also consistent with Drexler’s \ncomprehens

### Retreival

In [73]:
def retreiveFromVectorDB(query, k):
    response = collection_research.query(
    query_texts=[query],
    n_results=k,
    query_embeddings = [createEmbedding(query)],
    )
    return response["documents"][0]

def answer(question, k=4):
    chunks = retreiveFromVectorDB(question, k=k)
    CONTEXT = "\n\n---\n\n".join(chunks)  
    QUESTION = question
    prompt = f"""Answer the question using only the context below.
If the context does not contain the answer, say "I don't know based on the document."

Context:
{CONTEXT}

Question: {QUESTION}"""

    resp = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0},
    )
    return resp.message.content

In [74]:
print(answer("What is Transformative AI?"))

 Transformative AI, as defined in the context, is AI that significantly transforms society by replacing humans for a large portion (50% or greater) of economically useful work. This includes notions such as artificial general intelligence (AGI), high-level machine intelligence (HLMI), human-level machine intelligence, human-level artificial intelligence (HLAI), and machine superintelligence, as well as narrow AI technologies that meet this definition.
